In [ ]:
# Portable project paths (GitHub-friendly)
from pathlib import Path

CWD = Path.cwd()
ROOT = CWD if (CWD / 'saved_features').exists() else CWD.parent
FEATURE_DIR = ROOT / 'saved_features'
MODEL_PATH = ROOT / 'task1' / 'model.pth'
VALIDATION_DIR = ROOT / 'validation_data'

print('ROOT:', ROOT)
print('FEATURE_DIR exists:', FEATURE_DIR.exists())
print('MODEL_PATH exists:', MODEL_PATH.exists())


In [1]:
import os

for dirname, _, filenames in os.walk(str(ROOT)):
    print(dirname)
    for filename in filenames:
        print("   ", filename)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/sweetiswami
/kaggle/input/datasets/sweetiswami/ota-data
    test_features_full.npy
    test_labels.npy
    train_mean.npy
    val_features_full.npy
    val_labels.npy
    train_std.npy
    train_features_full.npy


In [2]:
import numpy as np

BASE = str(FEATURE_DIR) + "/"

train_features = np.load(BASE + "train_features_full.npy")
val_features   = np.load(BASE + "val_features_full.npy")
test_features  = np.load(BASE + "test_features_full.npy")

val_labels  = np.load(BASE + "val_labels.npy")
test_labels = np.load(BASE + "test_labels.npy")

mean = np.load(BASE + "train_mean.npy")
std  = np.load(BASE + "train_std.npy")

print(train_features.shape, val_features.shape, test_features.shape)

(862346, 2048) (300000, 2048) (384714, 2048)


In [3]:
train_features = (train_features - mean) / std
val_features   = (val_features   - mean) / std
test_features  = (test_features  - mean) / std

In [4]:
import torch
from torch.utils.data import Dataset

class TrainDataset(Dataset):
    def __init__(self, features, window=8, stride=5):
        self.features = features
        self.window = window
        self.stride = stride
        self.length = (len(features) - window) // stride

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        start = idx * self.stride
        seq = self.features[start:start+self.window]
        return torch.tensor(seq, dtype=torch.float32)

In [5]:
class TestDataset(Dataset):
    def __init__(self, features, labels, window=8, stride=5):
        self.features = features
        self.labels = labels
        self.window = window
        self.stride = stride
        self.length = (len(features) - window) // stride

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        start = idx * self.stride

        seq = self.features[start:start+self.window]
        label = 1 if np.any(self.labels[start:start+self.window]) else 0

        return torch.tensor(seq, dtype=torch.float32), label

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    TrainDataset(train_features),
    batch_size=128,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    TestDataset(val_features, val_labels),
    batch_size=128,
    num_workers=2
)

test_loader = DataLoader(
    TestDataset(test_features, test_labels),
    batch_size=128,
    num_workers=2
)

In [7]:
import torch.nn as nn

class TransformerAE(nn.Module):
    def __init__(self, input_dim=2048, d_model=256):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.decoder = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        return self.decoder(x)

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerAE().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

In [9]:
for epoch in range(15):
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        output = model(batch)
        loss = criterion(output, batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: {total_loss/len(train_loader):.4f}")

Epoch 0: 0.2472
Epoch 1: 0.1119
Epoch 2: 0.0872
Epoch 3: 0.0753
Epoch 4: 0.0681
Epoch 5: 0.0632
Epoch 6: 0.0598
Epoch 7: 0.0572
Epoch 8: 0.0553
Epoch 9: 0.0539
Epoch 10: 0.0528
Epoch 11: 0.0520
Epoch 12: 0.0515
Epoch 13: 0.0510
Epoch 14: 0.0507


In [10]:
from sklearn.metrics import roc_auc_score

def get_scores(model, loader):
    model.eval()
    scores = []
    labels = []

    with torch.no_grad():
        for batch, y in loader:
            batch = batch.to(device)

            output = model(batch)

            frame_mse = ((output - batch) ** 2).mean(dim=2)

            # 🔥 CRITICAL
            score = frame_mse.max(dim=1)[0]

            scores.extend(score.cpu().numpy())
            labels.extend(y.numpy())

    return np.array(scores), np.array(labels)

In [11]:
val_scores, val_y = get_scores(model, val_loader)
test_scores, test_y = get_scores(model, test_loader)

from sklearn.metrics import roc_auc_score

print("VAL AUC:", roc_auc_score(val_y, val_scores))
print("TEST AUC:", roc_auc_score(test_y, test_scores))

VAL AUC: 0.7322097078776825
TEST AUC: 0.828627983633997


In [12]:
torch.save(model.state_dict(), "model.pth")

In [13]:
model.load_state_dict(torch.load(str(MODEL_PATH)))

<All keys matched successfully>

In [14]:
model = TransformerAE()

model.load_state_dict(
    torch.load(str(MODEL_PATH), map_location="cpu")
)

model.eval()

TransformerAE(
  (input_proj): Linear(in_features=2048, out_features=256, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): Sequential(
    (0): Linear(in_features=256, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=2048, bias=True)
  )
)

In [15]:
x = torch.randn(1, 8, 2048)
print(model(x).shape)

torch.Size([1, 8, 2048])


In [16]:
!pip install onnx onnxruntime onnxruntime-tools onnxconverter-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.5 MB/s eta 0:00:00


In [17]:
!pip install onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 9.4 MB/s eta 0:00:00


In [18]:
import torch
import torch.nn as nn
import numpy as np
import time
import queue
import threading

from torchvision import models
import onnxruntime as ort

In [19]:
class TransformerAE(nn.Module):
    def __init__(self, input_dim=2048, d_model=256):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.decoder = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        return self.decoder(x)


model = TransformerAE()
model.load_state_dict(torch.load(str(MODEL_PATH), map_location="cpu"))
model.eval()

TransformerAE(
  (input_proj): Linear(in_features=2048, out_features=256, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): Sequential(
    (0): Linear(in_features=256, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=2048, bias=True)
  )
)

In [20]:
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()

dummy = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    resnet,
    dummy,
    "encoder.onnx",
    input_names=["image"],
    output_names=["embedding"],
    opset_version=18,   # 🔥 FIX
    dynamo=False
)

print("Encoder exported")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 159MB/s]
/tmp/ipykernel_24/3584046431.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Encoder exported


In [21]:
class TemporalWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        output = self.model(x)
        mse = ((output - x) ** 2).mean(dim=2)
        score = mse.max(dim=1)[0]
        return score


wrapper = TemporalWrapper(model)

dummy = torch.randn(1, 8, 2048)

torch.onnx.export(
    wrapper,
    dummy,
    "temporal_model.onnx",
    input_names=["sequence"],
    output_names=["score"],
    opset_version=14
)

print("Temporal exported")

/tmp/ipykernel_24/4110787375.py:17: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
W0325 21:59:28.035000 24 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0325 21:59:29.189000 24 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'b

[torch.onnx] Obtain model graph for `TemporalWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TemporalWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).
Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 2 of general pattern rewrite rules.
Temporal exported


In [22]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    "encoder.onnx",
    "encoder_int8.onnx",
    weight_type=QuantType.QInt8
)

print("Encoder quantized")

Encoder quantized


In [23]:
from onnxconverter_common import float16

import onnxruntime as ort

so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

sess = ort.InferenceSession("temporal_model.onnx", so)

In [24]:
encoder_sess = ort.InferenceSession("encoder_int8.onnx")
temporal_sess = ort.InferenceSession("temporal_model.onnx")

In [25]:
frame_queue = queue.Queue(maxsize=32)
feature_queue = queue.Queue(maxsize=32)

buffer = []

In [26]:
def encoder_worker():
    while True:
        frame = frame_queue.get()

        embedding = encoder_sess.run(
            None, {"image": frame}
        )[0]

        embedding = embedding.reshape(1, -1)  # (1, 2048)

        feature_queue.put(embedding)

In [27]:
def temporal_worker():
    global buffer

    while True:
        feat = feature_queue.get()
        buffer.append(feat)

        if len(buffer) >= 8:
            seq = np.concatenate(buffer[-8:], axis=0)
            seq = seq[np.newaxis, :, :]  # (1, 8, 2048)

            start = time.time()

            score = temporal_sess.run(
                None, {"sequence": seq.astype(np.float32)}
            )[0]

            latency = (time.time() - start) * 1000

            print(f"Score: {score[0]:.4f} | Latency: {latency:.2f} ms")

In [28]:
threading.Thread(target=encoder_worker, daemon=True).start()
threading.Thread(target=temporal_worker, daemon=True).start()

In [29]:
def push_frame(frame):
    if frame_queue.full():
        frame_queue.get()  # drop oldest

    frame_queue.put(frame)

In [30]:
# simulate frames
for _ in range(50):
    fake_frame = np.random.randn(1, 3, 224, 224).astype(np.float32)
    push_frame(fake_frame)

    time.sleep(0.1)  # simulate FPS

Score: 1.0879 | Latency: 5.64 ms
Score: 1.0874 | Latency: 2.37 ms
Score: 1.0875 | Latency: 5.17 ms
Score: 1.0726 | Latency: 2.23 ms
Score: 1.0721 | Latency: 2.16 ms
Score: 1.0717 | Latency: 1.81 ms
Score: 1.0643 | Latency: 1.62 ms
Score: 1.0649 | Latency: 1.57 ms
